In [42]:
import pandas as pd
import pandas as pd
import numpy as np
from scipy.stats import hmean

df_xgb = pd.read_csv('parallel_XGBoost_metrics.csv')
df_svr = pd.read_csv('parallel_svr_fitting_metrics.csv')
df_lstm = pd.read_csv('parallel_LSTM_metrics.csv')
df_gpr = pd.read_csv('parallel_gpr_metrics.csv')
df_ridge = pd.read_csv('parallel_ridge_regression_metrics.csv')
persistence_df = pd.read_csv('persistence_rmse.csv')

In [43]:
df_xgb

,Model,Training Time (s),Prediction Time (s),MAE,R-squared,RMSE,MSE
0,Parallel XGBoost,5.843524,295.790597,0.027439,0.646711,0.072297,0.005227


In [44]:
df_svr

,Model,Training Time (s),Prediction Time (s),MAE,R-squared,RMSE,MSE
0,Parallel SVR (Fitting & Prediction),10.35585,111.808022,0.060923,0.43664,0.091296,0.008335


In [45]:
df_lstm

,Model,Training Time (s),Prediction Time (s),MAE,R-squared,RMSE,MSE
0,Parallel LSTM,82.978783,66.458301,0.033274,0.80084,0.054282,0.002947


In [46]:
df_gpr

,Model,Training Time (s),Prediction Time (s),MAE,R-squared,RMSE,MSE
0,Parallel GPR (Fitting & Prediction),289.78429,949.229096,0.025957,0.709593,0.065548,0.004297


In [47]:
df_ridge

,Model,Training Time (s),Prediction Time (s),MAE,R-squared,RMSE,MSE
0,Parallel Ridge Regression,1.868803,2.487231,0.027289,0.722053,0.064127,0.004112


In [48]:
persistence_df

,Model,RMSE
0,Persistence Model,0.07137


In [49]:
final_df = pd.concat([df_xgb, df_svr, df_lstm, df_gpr, df_ridge], ignore_index=True)
final_df

,Model,Training Time (s),Prediction Time (s),MAE,R-squared,RMSE,MSE
0,Parallel XGBoost,5.843524,295.790597,0.027439,0.646711,0.072297,0.005227
1,Parallel SVR (Fitting & Prediction),10.355850,111.808022,0.060923,0.436640,0.091296,0.008335
2,Parallel LSTM,82.978783,66.458301,0.033274,0.800840,0.054282,0.002947
3,Parallel GPR (Fitting & Prediction),289.784290,949.229096,0.025957,0.709593,0.065548,0.004297
4,Parallel Ridge Regression,1.868803,2.487231,0.027289,0.722053,0.064127,0.004112


In [50]:
final_df.drop(columns=['MAE', 'MSE'], axis='columns', inplace=True)
final_df

,Model,Training Time (s),Prediction Time (s),R-squared,RMSE
0,Parallel XGBoost,5.843524,295.790597,0.646711,0.072297
1,Parallel SVR (Fitting & Prediction),10.355850,111.808022,0.436640,0.091296
2,Parallel LSTM,82.978783,66.458301,0.800840,0.054282
3,Parallel GPR (Fitting & Prediction),289.784290,949.229096,0.709593,0.065548
4,Parallel Ridge Regression,1.868803,2.487231,0.722053,0.064127


In [51]:
rmse_value = persistence_df.loc[0, 'RMSE']
rmse_value


0.0713702957793329

In [52]:
final_df['skill_score'] = 1 - (final_df['RMSE'] / rmse_value)
final_df

,Model,Training Time (s),Prediction Time (s),R-squared,RMSE,skill_score
0,Parallel XGBoost,5.843524,295.790597,0.646711,0.072297,-0.012990
1,Parallel SVR (Fitting & Prediction),10.355850,111.808022,0.436640,0.091296,-0.279185
2,Parallel LSTM,82.978783,66.458301,0.800840,0.054282,0.239427
3,Parallel GPR (Fitting & Prediction),289.784290,949.229096,0.709593,0.065548,0.081574
4,Parallel Ridge Regression,1.868803,2.487231,0.722053,0.064127,0.101494


In [53]:
ranking_df = pd.DataFrame()
ranking_df["Model"] = final_df["Model"]

for col in ["Training Time (s)", "Prediction Time (s)",  "RMSE"]:
    ranking_df[col] = final_df[col].rank(method="min", ascending=True).astype(int)

ranking_df["R-squared"] = final_df["R-squared"].rank(method="min", ascending=False).astype(int) 
ranking_df["skill_score"] = final_df["skill_score"].rank(method="min", ascending=False).astype(int) 

ranking_df

,Model,Training Time (s),Prediction Time (s),RMSE,R-squared,skill_score
0,Parallel XGBoost,2,4,4,4,4
1,Parallel SVR (Fitting & Prediction),3,3,5,5,5
2,Parallel LSTM,4,2,1,1,1
3,Parallel GPR (Fitting & Prediction),5,5,3,3,3
4,Parallel Ridge Regression,1,1,2,2,2


In [54]:
ranking_df["Row Average"] = ranking_df.iloc[:, 1:].mean(axis=1)

In [55]:
ranking_df.sort_values(by=['Row Average'])

,Model,Training Time (s),Prediction Time (s),RMSE,R-squared,skill_score,Row Average
4,Parallel Ridge Regression,1,1,2,2,2,1.6
2,Parallel LSTM,4,2,1,1,1,1.8
0,Parallel XGBoost,2,4,4,4,4,3.6
3,Parallel GPR (Fitting & Prediction),5,5,3,3,3,3.8
1,Parallel SVR (Fitting & Prediction),3,3,5,5,5,4.2


**Min-Max Scaling**

In [56]:
def custom_minmax_normalization(df, higher_is_better, cutoff=-0.5):
    df_norm = pd.DataFrame(index=df.index)

    for col in df.columns:
        norm = df[col].copy()
        if col != 'Model':
            x = df[col].copy()

            x = x.clip(lower=cutoff)

            if higher_is_better[col]:
                norm = (x - x.min()) / (x.max() - x.min())
            else:
                norm = (x.max() - x) / (x.max() - x.min())

        df_norm[col] = norm

    return df_norm

def nested_harmonic_score(df, higher_is_better, cutoff=-0.5):
    df_norm = custom_minmax_normalization(df, higher_is_better, cutoff)

    #Performance Score
    df_norm["Performance_Score"] = df_norm[["R-squared", "RMSE", "skill_score"]].apply(
        lambda row: hmean(row) if all(row > 0) else 0, axis=1
    )

    #Speed Score
    df_norm["Speed_Score"] = df_norm[["Training Time (s)", "Prediction Time (s)"]].apply(
        lambda row: hmean(row) if all(row > 0) else 0, axis=1
    )

    #Overall Score
    df_norm["Overall_Score"] = df_norm[["Performance_Score", "Speed_Score"]].apply(
        lambda row: hmean(row) if all(row > 0) else 0, axis=1
    )

    return df_norm[["Model", "Performance_Score", "Speed_Score", "Overall_Score"]]


higher_is_better = {
    "Training Time (s)": False,
    "Prediction Time (s)": False,
    "R-squared": True,
    "RMSE": False,
    "skill_score": True,
}

result = nested_harmonic_score(final_df, higher_is_better)
result.sort_values("Overall_Score", ascending=False, inplace=True)
result


,Model,Performance_Score,Speed_Score,Overall_Score
2,Parallel LSTM,1.000000,0.811468,0.895923
4,Parallel Ridge Regression,0.749867,1.000000,0.857056
0,Parallel XGBoost,0.532843,0.812064,0.643468
1,Parallel SVR (Fitting & Prediction),0.000000,0.925533,0.000000
3,Parallel GPR (Fitting & Prediction),0.712689,0.000000,0.000000
